In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Delta_Project").getOrCreate()

data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]

columns = ["id", "customer_id", "product", "amount"]

df = spark.createDataFrame(data, columns)

TASK 1: Create Delta Table

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("orders")

TASK 2: Insert New Data

In [0]:
new_data = [(5, "C005", "Camera", 30000)]
new_df = spark.createDataFrame(new_data, columns)

new_df.write.format("delta").mode("append").saveAsTable("orders")
spark.table("orders").show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  1|       C001| Laptop| 50000|
|  2|       C002| Mobile| 15000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
|  5|       C005| Camera| 30000|
|  5|       C005| Camera| 30000|
+---+-----------+-------+------+



TASK 3: Update Data

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "orders")

delta_table.update(
    condition = "id = 2",
    set = {"amount": "18000"}
)
spark.table("orders").show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  2|       C002| Mobile| 18000|
|  1|       C001| Laptop| 50000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
|  5|       C005| Camera| 30000|
|  5|       C005| Camera| 30000|
+---+-----------+-------+------+



TASK 4: Delete Data

In [0]:
delta_table.delete("id = 1")
spark.table("orders").show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
|  2|       C002| Mobile| 18000|
|  5|       C005| Camera| 30000|
|  5|       C005| Camera| 30000|
+---+-----------+-------+------+



TASK 5: MERGE (UPSERT)

In [0]:
updates = [
    (3, "C003", "Tablet", 22000),  # update
    (6, "C006", "Watch", 8000)     # insert
]

updates_df = spark.createDataFrame(updates, columns)

delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "customer_id": "source.customer_id",
    "product": "source.product",
    "amount": "source.amount"
}).whenNotMatchedInsert(values={
    "id": "source.id",
    "customer_id": "source.customer_id",
    "product": "source.product",
    "amount": "source.amount"
}).execute()
spark.table("orders").show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  4|       C004| Laptop| 55000|
|  2|       C002| Mobile| 18000|
|  5|       C005| Camera| 30000|
|  5|       C005| Camera| 30000|
|  3|       C003| Tablet| 22000|
|  6|       C006|  Watch|  8000|
+---+-----------+-------+------+



TASK 6: Schema Evolution (Add Column)

In [0]:
from pyspark.sql.functions import lit

df_new = spark.table("orders") \
    .withColumn("country", lit("India"))

df_new.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("orders")
spark.table("orders").show()

+---+-----------+-------+------+-------+
| id|customer_id|product|amount|country|
+---+-----------+-------+------+-------+
|  4|       C004| Laptop| 55000|  India|
|  2|       C002| Mobile| 18000|  India|
|  5|       C005| Camera| 30000|  India|
|  5|       C005| Camera| 30000|  India|
|  3|       C003| Tablet| 22000|  India|
|  6|       C006|  Watch|  8000|  India|
+---+-----------+-------+------+-------+



TASK 7: Time Travel

In [0]:
spark.sql("DESCRIBE HISTORY orders").show(truncate=False)
spark.table("orders").display()

+-------+-------------------+--------------+--------------------------+---------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

id,customer_id,product,amount,country
4,C004,Laptop,55000,India
2,C002,Mobile,18000,India
5,C005,Camera,30000,India
5,C005,Camera,30000,India
3,C003,Tablet,22000,India
6,C006,Watch,8000,India


In [0]:
old_df = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table("orders")

old_df.show()

+--------+-------------+----------+---------------+-------------+------------+
|order_id|customer_name|order_date|order_timestamp|delivery_date|order_amount|
+--------+-------------+----------+---------------+-------------+------------+
+--------+-------------+----------+---------------+-------------+------------+



In [0]:
delta_table.restoreToVersion(0)
spark.table("orders").display()

order_id,customer_name,order_date,order_timestamp,delivery_date,order_amount
